In [12]:
import plotly.graph_objects as go
import json
import pandas as pd
import numpy as np

In [13]:
base_path = "../finetuned_results"
score_path = "../finetuned_results"
split = "test"
template = "Instruct-Query"
use_lang_specific_prompts=False
k = 10
models = [
          "Qwen__Qwen3-Embedding-0.6B",
          #"__flash__project_462001491__models__v1-20260828-095152__checkpoint-2000",
          #"__flash__project_462001491__models__v1-20260828-095152__checkpoint-4000",
          #"__flash__project_462001491__models__v1-20260828-095152__checkpoint-6000",
          #"__flash__project_462001491__models__v1-20260828-095152__checkpoint-12000",
          "__flash__project_462001491__models__v1-20260828-095152__checkpoint-18000"
          ]

dataset = "mteb__ARCChallenge" #"tatoeba:fra-eng" #"mteb__tatoeba-bitext-mining:fin-eng" #"mteb__ARCChallenge"#"mteb__multi-hatecheck:eng" #"mteb__ARCChallenge" #"mteb__tatoeba-bitext-mining:ara-eng" #"mteb__multi-hatecheck:eng" #"mteb__reddit-clustering" #"mteb__stsbenchmark-sts" #"mteb__tatoeba-bitext-mining:fin-eng"
score= "ndcg@10" # "ndcg@10"#"F1" #"Accuracy" #"V-score" # "average_precision" 
subsplit=""
if dataset == "mteb__reddit-clustering":
    subsplit="0"
path = lambda model: f"{base_path}/{model}/{dataset}/{split}/{template}_template/"
path_scores = lambda model: f"{score_path}/{model}/{dataset}/{split}/{template}_template/"

In [14]:

def construct_df(model, show=False):
    scores_path= path_scores(model)+f"results@{k}.json"
    with open(scores_path) as f:
        scores = json.load(f)
    with open(path(model)+f"prompt_geometry{subsplit}.json") as f:
        data1 = json.load(f)
    if show:
        print(data1.keys())
        #print(data1)
    #print(scores)
    df_scores = pd.DataFrame.from_dict(scores).T#, orient="index", columns=["score"])
    #columns= [f"prompt{i}" for i in range(len(scores.values()))])
    #df_scores = df_scores.reset_index().rename(columns={"index": "prompt_text"})#, "mean":"score_mean", "std":"score_std"})
    if show: display(df_scores.head())
    #df_scores["score_mean"] = pd.to_numeric(df_scores["score_mean"])
    df_angle = pd.DataFrame.from_dict(data1).T
    #display(df_scores.head())
    if show: display(df_angle.head())
    df = df_scores.merge(df_angle, on='prompt_text')
    #prompt_dict = prompts(dataset)
    #df["prompt_label"] = df["prompt_text"].apply(lambda prompt: prompt_dict[prompt])
    if show: display(df.head())
    return df

#_ = construct_df(models[0], show=False)

In [15]:

def plot(df, x, y="score", colors=None, sizes=None, title="", legend_title=None, x_min=None, x_max=None, y_min=None, y_max=None, return_fig=False):
    if colors is None:
        colors = y
    if sizes is None:
        sizes = y
    
    # legend title that explains formatting
    if legend_title is None:
        legend_title = f"colors:{colors}, size:{sizes}"

    # Normalize scores for marker size
    min_size, max_size = 10, 30
    try:
        # parse the value from dictionary
        df["sizes"] = df[sizes].apply(lambda d: float(d['mean']))
        ranks = df["sizes"].rank(method='average')
    except:   # for non-dict format: i.e. prompt_label or score
        ranks = df[sizes].rank(method='average')
    marker_sizes = (
        (ranks - ranks.min()) / (ranks.max() - ranks.min()) * (max_size - min_size) + min_size
    )

    y_vals = df[y].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
    #y_err = df[y].apply(lambda d: float(d['std']) if isinstance(d, dict) else float(d[1]) if isinstance(d, list) else 0.0)
    y_err = []
    for line in df[y]:
        if isinstance(line, dict):
            if "std" in line.keys():
                y_err.append(float(line["std"]))
            elif "confidence_interval" in line.keys():
                y_err.append(float(max(line["confidence_interval"])))
            else:
                y_err.append(0.0)
        else:
            y_err.append(0.0)
        

    x_vals = df[x].apply(lambda d: float(d['mean']))
    x_err  = df[x].apply(lambda d: float(d['std']))


    # Create figure
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            mode='markers',
            x=x_vals,
            y=y_vals,
            #error_x=dict(type='data', array=x_err, visible=True, color='lightgray'),  # optional std bars
            #error_y=dict(type='data', array=y_err, visible=True, color='lightgray'),
            marker=dict(
                size=marker_sizes,
            #    colorscale='Cividis',
                color=df[colors].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d)),
                colorbar=dict(title=f"C:{colors} S:{sizes}"),
                showscale=True,
            ),
            text=df['prompt_text'],
            hovertemplate=(
                '<b>Prompt:</b> %{text}<br>'
                '<b>X (distance):</b> %{x:.4f}<br>'
                '<b>Y (score):</b> %{y:.4f}<br>'
            ),
        ),
    )


    # Add one trace per alpha value

    fig.update_layout(
        title = title,
        xaxis_title=x,#'Cos-distance compared to Q-A line',
        yaxis_title=y,#'Prompt performance',
        height=600,
        width=1000,
        template="none",
    )
    fig.update_layout(legend_title_text=legend_title)
    fig.update_xaxes(range=[x_min, x_max])
    fig.update_yaxes(range=[y_min, y_max], autorange=False)

    if return_fig:
        return fig
    else:
        fig.show()

In [16]:
dfs = {}
for m in models:
    try:
        dfs[m] = construct_df(m)
    except Exception as e:
        print(f"Cannot construct results for {m}")
        print(e)

In [17]:

for m in dfs.keys():
    df = dfs[m]
    #print(df.columns)
    #plot(df, "displacement", y=score, title=f"{dataset}: {m}: Does more movement(x) mean better score(y)")
    #plot(df, "sim_improvement", y=score, title=f"{dataset}: {m}: Does sim-improvement (x) actually mean better score? (y)")
    #plot(df, "chord_similarity", y=score, title=f"{dataset}: {m}: Fraction of change toward Answer (x) vs. performance(y).")
    #plot(df, "knn_retention", y=score,title=f"{dataset}: {m}: Neighborhood retention (x) vs. performance (y)")
    #plot(df, "orthogonal_magnitude", y=score, title=f"{dataset}: {m}:")
    #plot(df, "hard_neg_angulation", y="chord_similarity", colors=score, sizes=score, title=f"{dataset}: {m}: Angle from false negs vs. evaluation score")
    #plot(df, "hard_neg_sim_change", y=score, title=f"{dataset}: {m}: Movement away from false negs vs. evaluation score")#, sizes="sim_improvement", colors="sim_improvement")
    
	# combinations
    #plot(df, "hard_neg_sim_change", y="sim_improvement", sizes="score", colors="knn_retention",title=f"{dataset}: {m}:")
    #plot(df, "hard_neg_angulation", y="sim_improvement", sizes="score", colors="score",title=f"{dataset}: {m}: Angle from false negs vs. sim improvement")
    #plot(df, "displacement", y="sim_improvement", colors="knn_retention", sizes="knn_retention", title=f"{dataset}: {m}: Movement (x) vs. similarity improvement(y).")
    #plot(df, "parallel_fraction", y="sim_improvement", title=f"{dataset}: {m}: Fraction of change toward Answer (x) vs. similarity gains.")
    #plot(df, "displacement", y="parallel_fraction", sizes="score", colors="prompt_label", title=f"{dataset}: {m}: Does more movement(x) mean better score(y)")


    # this is interesting for sem sim
    #plot(df, "chord_similarity", y="orthogonal_magnitude", colors=score, sizes=score)


In [18]:
# Retrieval prompt
prompts = ["Given a question, retrieve the passage that best answers it.",
            "Retrieve.",
            "Find the most relevant passage that directly answers the question.",
            "Given a question, find a related document.",
            "Retrieve the answer to the question.",
            "Retrieve text based on user query.",
            "Given a question, retrieve Wikipedia passages that answer the question.",
            ]

In [19]:



def plot_paired_differences(
    df1: pd.DataFrame,
    df2: pd.DataFrame,
    hover_col: str = None,
    df1_name: str = "Group A",
    df2_name: str = "Group B",
    df1_color: str = "royalblue",
    df2_color: str = "tomato",
    line_color: str = "gray",
    x="displacement",
    y="ndcg@10"
):
    """
    Plot paired (x, y) points from two DataFrames, connected by dashed lines.

    Parameters
    ----------
    df1, df2    : DataFrames with columns 'x' and 'y' (same length, rows are paired)
    hover_col   : Optional column name in both DataFrames to show as hover text
    df1_name    : Legend label for df1 points
    df2_name    : Legend label for df2 points
    df1_color   : Marker color for df1
    df2_color   : Marker color for df2
    line_color  : Color of the dashed connector lines
    """
    assert len(df1) == len(df2), "DataFrames must have the same number of rows."

    fig = go.Figure()
    # for some data, we need to parse the column (from dict or tuple)
    y_vals1 = df1[y].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
    y_vals2 = df2[y].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
    x_vals1 = df1[x].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
    x_vals2 = df2[x].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))

    # Connectroe lines
    # Interleave (x1, x2, None) for each pair so plotly draws separate segments
    line_x, line_y = [], []
    for x1, x2, y1, y2 in zip(x_vals1, x_vals2, y_vals1, y_vals2):
        line_x += [x1, x2, None]
        line_y += [y1, y2, None]

    fig.add_trace(go.Scatter(
        x=line_x,
        y=line_y,
        mode="lines",
        line=dict(dash="dash", color=line_color, width=1.5),
        hoverinfo="skip",
        showlegend=False,
    ))

    # Hover template
    def make_hover(df, group_name):
        if hover_col and hover_col in df.columns:
            return (
                df[hover_col].tolist(),
                f"<b>{group_name}</b><br>"
                f"x: %{{x}}<br>y: %{{y}}<br>"
                f"{hover_col}: %{{text}}<extra></extra>",
            )
        return (None, f"<b>{group_name}</b><br>x: %{{x}}<br>y: %{{y}}<extra></extra>")
    
    # First data
    colors1 = [df1_color if p in prompts else "lightblue" for p in df1["prompt_text"]]
    text1, htemplate1 = make_hover(df1, df1_name)
    fig.add_trace(go.Scatter(
        x=x_vals1, y=y_vals1,
        mode="markers",
        name=df1_name,
        marker=dict(color=colors1, size=10, line=dict(width=1, color="white")),
        text=text1,
        hovertemplate=htemplate1,
    ))

    # Second data
    colors2 = [df2_color if p in prompts else "lightpink" for p in df2["prompt_text"]]
    text2, htemplate2 = make_hover(df2, df2_name)
    fig.add_trace(go.Scatter(
        x=x_vals2, y=y_vals2,
        mode="markers",
        name=df2_name,
        marker=dict(color=colors2, size=10, line=dict(width=1, color="white")),
        text=text2,
        hovertemplate=htemplate2,
    ))

    fig.update_layout(
        xaxis_title=x,
        yaxis_title=y,
        height=600,
        width=1000,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        template="plotly_white",
    )

    #fig.show()
    return fig


In [20]:
def plot_difference(
    df1: pd.DataFrame,
    df2: pd.DataFrame,
    hover_col: str = None,
    df1_name: str = "Group A",
    df2_name: str = "Group B",
    df1_color: str = "royalblue",
    df2_color: str = "tomato",
    line_color: str = "gray",
    x="displacement",
    y="ndcg@10",
    sizes="ndcg@10",
):
    
    assert len(df1) == len(df2), "DataFrames must have the same number of rows."

    fig = go.Figure()
    # for some data, we need to parse the column (from dict or tuple)
    y_vals1 = df1[y].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
    y_vals2 = df2[y].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
    x_vals1 = df1[x].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
    x_vals2 = df2[x].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
    x_vals_diff = np.array(x_vals2)-np.array(x_vals1)
    y_vals_diff = np.array(y_vals2)-np.array(y_vals1)
    
    # Hover template
    def make_hover(df, group_name):
        if hover_col and hover_col in df.columns:
            return (
                df[hover_col].tolist(),
                f"<b>{group_name}</b><br>"
                f"x: %{{x}}<br>y: %{{y}}<br>"
                f"{hover_col}: %{{text}}<extra></extra>",
            )
        return (None, f"<b>{group_name}</b><br>x: %{{x}}<br>y: %{{y}}<extra></extra>")

    # Normalize scores for marker size
    min_size, max_size = 8, 32
    df1["sizes"] = df1[sizes].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
    ranks = df1["sizes"].rank(method='average')
    marker_sizes = (
        (ranks - ranks.min()) / (ranks.max() - ranks.min()) * (max_size - min_size) + min_size
    )
    # First data
    colors = [df2_color if p in prompts else "lightpink" for p in df2["prompt_text"]]
    text, htemplate = make_hover(df1, df1_name)
    fig.add_trace(go.Scatter(
        x=x_vals_diff, y=y_vals_diff,
        mode="markers",
        name="Difference",
        marker=dict(color=colors, size=marker_sizes, line=dict(width=1, color="white")),
        text=text,
        hovertemplate=htemplate,
    ))


    fig.update_layout(
        xaxis_title=f"Difference in {x}",
        yaxis_title=f"Difference in {y}",
        height=600,
        width=1000,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        template="plotly_white",
    )

    #fig.show()
    return fig

In [21]:
x = "hard_neg_angulation"
y = "ndcg@10"

fig = plot_paired_differences(
                            dfs["Qwen__Qwen3-Embedding-0.6B"], 
                            #dfs["__flash__project_462001491__models__v1-20260828-095152__checkpoint-2000"], 
                            dfs["__flash__project_462001491__models__v1-20260828-095152__checkpoint-18000"], 
                            hover_col="prompt_text", 
                            x=x,
                            y=y,
                            df1_name="Qwen3-Embedding-0.6B", 
                            df2_name="Finetuning checkpoint 18k")
fig.show()

fig = plot_difference(
                    dfs["Qwen__Qwen3-Embedding-0.6B"], 
                    #dfs["__flash__project_462001491__models__v1-20260828-095152__checkpoint-2000"], 
                    dfs["__flash__project_462001491__models__v1-20260828-095152__checkpoint-18000"],
                    hover_col="prompt_text",
                    x =x,
                    y=y)
fig.show()